# Module 3 — Lab C: Exploratory Data Analysis & Feature Engineering
## Truck Delay Classification — FreshBasket Logistics

---

### Lab Overview

FreshBasket's logistics team (led by **Priya**) needs to predict whether a truck delivery will be delayed. The raw data lives in **seven PostgreSQL tables** you loaded into RDS in Lab B. In this lab you will connect to that database, explore each table, merge them into a single analysis-ready feature matrix, and save it for model training in Lab D.

The feature matrix has **36 features** (27 continuous + 9 categorical) and one binary target (`delay`). Building it requires careful aggregation of 2.6 million traffic rows and 480,000+ weather rows down to daily route-level summaries — the kind of data wrangling that is routine in production ML pipelines.

| Step | What We Do | Key Technique |
|------|-----------|---------------|
| 1 | Connect to RDS PostgreSQL | SQLAlchemy, psycopg2 |
| 2 | Load and profile each table | Shape, dtypes, nulls, distributions |
| 3 | Analyse the target variable | Class balance, delay rate |
| 4 | Merge core tables | Left joins from schedule backbone |
| 5 | Aggregate traffic data | Daily avg vehicles + max accident per route |
| 6 | Aggregate weather data | Route, origin city, destination city weather |
| 7 | Engineer temporal features | Hour extraction, is_midnight flag |
| 8 | Visualise relationships (EDA) | Correlation heatmap, bar charts, distributions |
| 9 | Assemble final feature matrix | 36 features + target, null check |
| 10 | Save artifacts to local + S3 | CSV, JSON metadata |

### Learning Objectives

By the end of this lab you will be able to:
1. Connect a Jupyter notebook (SageMaker or local) to an RDS PostgreSQL instance
2. Profile multi-table datasets and identify join keys
3. Aggregate high-cardinality time-series tables (traffic, weather) to meaningful daily summaries
4. Engineer temporal features (hour-of-day, midnight flag) from datetime columns
5. Build a flat feature matrix from normalised relational tables
6. Save artifacts with metadata for downstream consumption

### Artifact Chain

```
Lab B (input)  -->  7 tables in RDS PostgreSQL (truck_delay_db)
                    S3 bucket with raw CSVs
Lab C (output) -->  data/processed/final_features.csv
                    data/processed/feature_metadata.json
                    (consumed by Lab D: Model Training & MLflow)
```

### Environment

| Environment | How to Connect to RDS |
|---|---|
| **SageMaker Notebook** (same VPC) | Use RDS endpoint directly — no tunnel needed |
| **Local Jupyter** | Requires SSH tunnel through EC2, or temporarily enable RDS public access |

---

## 1. Environment Setup

> **SageMaker Notebook:** Run the cell below as-is (psycopg2-binary and boto3 are pre-installed).
> **Local Jupyter:** Uncomment the pip install line if packages are not installed.

In [ ]:
# ---- Environment Setup ----
# Uncomment the line below if running locally and packages are not installed
# !pip install pandas numpy matplotlib seaborn psycopg2-binary sqlalchemy boto3 -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from sqlalchemy import create_engine
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('All libraries imported successfully')
print(f'  pandas:     {pd.__version__}')
print(f'  numpy:      {np.__version__}')
print(f'  Python env: 3.12.10 (.venv)')

## 2. Connect to RDS PostgreSQL

Replace the placeholder values below with your actual RDS endpoint and password from Lab B.

> **Security note:** In production, credentials go in AWS Secrets Manager or environment variables — never hard-coded. For this lab, we hard-code them for simplicity.

In [ ]:
# ---- Database Configuration ----
# Replace these with your actual values from Lab B
DB_CONFIG = {
    'host': '<YOUR_RDS_ENDPOINT>',       # e.g., truck-delay-db.c9xxxxxx.ap-south-1.rds.amazonaws.com
    'port': 5432,
    'database': 'truck_delay_db',
    'user': 'mlops_admin',
    'password': '<YOUR_PASSWORD>'         # In production, use AWS Secrets Manager
}

# S3 bucket for saving artifacts
S3_BUCKET = 'mlops-truck-delay-<your-name>-2026'   # Replace with your bucket name

# Local output directory
OUTPUT_DIR = 'data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}/')

In [ ]:
# ---- Create Database Connection ----
def get_db_engine(config):
    """Create a SQLAlchemy engine for PostgreSQL connection.

    Args:
        config: dict with host, port, database, user, password

    Returns:
        SQLAlchemy Engine object
    """
    conn_str = (
        f"postgresql://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/{config['database']}"
    )
    engine = create_engine(conn_str)

    # Test the connection
    with engine.connect() as conn:
        result = conn.execute(pd.io.sql.text("SELECT 1"))
        result.fetchone()

    print(f"Connected to: {config['host']}:{config['port']}/{config['database']}")
    return engine

engine = get_db_engine(DB_CONFIG)

## 3. Load and Profile Each Table

We have **7 tables** in the `truck_delay_db` database. Let's load each one and understand its structure before merging.

| Table | Rows | Role |
|---|---|---|
| `truck_schedule_table` | 12,308 | Backbone — one row per shipment, contains the **target** (`delay`) |
| `trucks_table` | 1,301 | Vehicle characteristics |
| `drivers_table` | 1,301 | Driver demographics and behaviour |
| `routes_table` | 2,353 | Route distances and average hours |
| `traffic_table` | 2,597,914 | Hourly traffic density per route |
| `city_weather` | 55,177 | Hourly weather per city (origin / destination) |
| `routes_weather` | 425,713 | Hourly weather per route |

In [ ]:
# ---- Helper: Load and Profile a Table ----
def load_and_profile(table_name, engine):
    """Load a PostgreSQL table into a DataFrame and print a profile summary.

    Args:
        table_name: name of the table in the database
        engine: SQLAlchemy engine

    Returns:
        pd.DataFrame
    """
    df = pd.read_sql(f"SELECT * FROM {table_name}", engine)

    print("=" * 60)
    print(f"TABLE: {table_name}")
    print("=" * 60)
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"\nColumns and dtypes:")
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = null_count / len(df) * 100
        null_str = f"  ({null_count} nulls, {null_pct:.1f}%)" if null_count > 0 else ""
        print(f"  {col:30s} {str(df[col].dtype):10s}{null_str}")
    print()
    return df

# ---- Load the backbone table first ----
schedule_df = load_and_profile('truck_schedule_table', engine)
schedule_df.head(3)

In [ ]:
# ---- Load trucks and drivers ----
trucks_df = load_and_profile('trucks_table', engine)
display(trucks_df.head(3))

print()
drivers_df = load_and_profile('drivers_table', engine)
display(drivers_df.head(3))

In [ ]:
# ---- Load routes ----
routes_df = load_and_profile('routes_table', engine)
display(routes_df.head(3))

In [ ]:
# ---- Load traffic (large table - 2.6M rows) ----
# This may take 15-30 seconds depending on network speed
print("Loading traffic_table (2.6M rows) -- please wait...")
traffic_df = load_and_profile('traffic_table', engine)
display(traffic_df.head(3))

print(f"\nDescriptive statistics:")
display(traffic_df.describe())

In [ ]:
# ---- Load city_weather ----
city_weather_df = load_and_profile('city_weather', engine)
display(city_weather_df.head(3))

In [ ]:
# ---- Load routes_weather ----
routes_weather_df = load_and_profile('routes_weather', engine)

# Standardise column name: 'Date' -> 'date' (case consistency)
if 'Date' in routes_weather_df.columns:
    routes_weather_df = routes_weather_df.rename(columns={'Date': 'date'})
    print("Renamed 'Date' -> 'date' for consistency")

display(routes_weather_df.head(3))

## 4. Target Variable Analysis

The `delay` column in `truck_schedule_table` is our binary target:
- **0** = On time
- **1** = Delayed

Understanding the class balance is critical — it determines whether we need stratified sampling or class weighting during model training.

In [ ]:
# ---- Class Balance of delay ----
delay_counts = schedule_df['delay'].value_counts()
delay_pct = schedule_df['delay'].value_counts(normalize=True) * 100

print("=== Target Variable: delay ===")
print(f"  0 (On time):  {delay_counts[0]:,} ({delay_pct[0]:.1f}%)")
print(f"  1 (Delayed):  {delay_counts[1]:,} ({delay_pct[1]:.1f}%)")
print(f"  Total:        {len(schedule_df):,}")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
bars = axes[0].bar(['On Time (0)', 'Delayed (1)'], delay_counts.values,
                    color=['#2563eb', '#f43f5e'], edgecolor='white', width=0.5)
axes[0].set_title('Shipment Delay Distribution', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Count')
for bar, count, pct in zip(bars, delay_counts.values, delay_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=11)

# Pie chart
axes[1].pie(delay_counts.values, labels=['On Time', 'Delayed'],
            colors=['#2563eb', '#f43f5e'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Proportions', fontweight='bold', fontsize=14)

plt.suptitle('FreshBasket Truck Delivery Delays', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Business insight
if delay_pct[1] > 40:
    print(f"\nInsight: {delay_pct[1]:.0f}% delay rate is significant.")
    print("Priya's team estimates each delayed shipment costs FreshBasket approx 2,500 rupees in spoilage and penalties.")
    print(f"Rough annual cost: approx {delay_counts[1] * 2500 / 100000:.1f} lakhs")

## 5. Merge Core Tables

**Merge strategy:** The `truck_schedule_table` is our backbone (one row per shipment). We left-join other tables onto it, preserving the 12,308-row structure.

```
truck_schedule_table (12,308 rows)
    |-- LEFT JOIN trucks_table        ON truck_id
    |-- LEFT JOIN drivers_table       ON truck_id = vehicle_no
    |-- LEFT JOIN routes_table        ON route_id
```

**Why left joins?** We never want to lose schedule rows. If a truck or driver record is missing, we keep the schedule row and handle the null later.

**Why drivers join on `vehicle_no`?** The `drivers_table` uses `vehicle_no` (not `truck_id`) as the link to trucks. Each driver is assigned to a specific vehicle.

In [ ]:
# ---- Step 1: Parse dates in the schedule table ----
schedule_df['departure_date'] = pd.to_datetime(schedule_df['departure_date'])
schedule_df['estimated_arrival'] = pd.to_datetime(schedule_df['estimated_arrival'])

# Extract the date portion for joining with traffic/weather (which are date-level)
schedule_df['depart_date'] = schedule_df['departure_date'].dt.date.astype(str)

print(f"Schedule table: {schedule_df.shape}")
print(f"Date range: {schedule_df['departure_date'].min()} to {schedule_df['departure_date'].max()}")
print(f"Unique trucks: {schedule_df['truck_id'].nunique()}")
print(f"Unique routes: {schedule_df['route_id'].nunique()}")

In [ ]:
# ---- Step 2: Merge schedule + trucks ----
df_merged = pd.merge(
    schedule_df,
    trucks_df,
    on='truck_id',
    how='left'
)
print(f"After merge with trucks_table:  {df_merged.shape}")
assert df_merged.shape[0] == schedule_df.shape[0], "Row count changed -- unexpected!"
print(f"  truck_age nulls:          {df_merged['truck_age'].isnull().sum()}")
print(f"  load_capacity nulls:      {df_merged['load_capacity_pounds'].isnull().sum()}")

# ---- Step 3: Merge + drivers ----
# drivers_table links via vehicle_no = truck_id
df_merged = pd.merge(
    df_merged,
    drivers_df,
    left_on='truck_id',
    right_on='vehicle_no',
    how='left'
)
print(f"\nAfter merge with drivers_table: {df_merged.shape}")
assert df_merged.shape[0] == schedule_df.shape[0], "Row count changed -- check for duplicates!"
print(f"  driver age nulls:         {df_merged['age'].isnull().sum()}")
print(f"  experience nulls:         {df_merged['experience'].isnull().sum()}")

# ---- Step 4: Merge + routes ----
df_merged = pd.merge(
    df_merged,
    routes_df,
    on='route_id',
    how='left'
)
print(f"\nAfter merge with routes_table:  {df_merged.shape}")
assert df_merged.shape[0] == schedule_df.shape[0], "Row count changed -- check for duplicates!"
print(f"  distance nulls:           {df_merged['distance'].isnull().sum()}")
print(f"  average_hours nulls:      {df_merged['average_hours'].isnull().sum()}")

print(f"\nCore merge complete. Columns ({df_merged.shape[1]}):")
print(f"  {df_merged.columns.tolist()}")

## 6. Aggregate Traffic Data

The `traffic_table` has **2.6 million rows** — one per route per hour. We cannot merge it directly (it would explode our 12,308 rows into millions). Instead, we aggregate to **daily averages per route**.

**Aggregation logic:**
- `avg_no_of_vehicles` = mean hourly vehicle count for that route on that date
- `accident` = max accident flag for that route on that date (1 if any accident that day)

```
traffic_table (2.6M rows, hourly)
    --> groupby(route_id, date).agg(mean vehicles, max accident)
    --> ~daily traffic summary per route
    --> LEFT JOIN onto merged dataset on (route_id, depart_date)
```

In [ ]:
# ---- Parse traffic date and aggregate to daily level ----
traffic_df['date'] = pd.to_datetime(traffic_df['date'])
traffic_df['date_str'] = traffic_df['date'].dt.date.astype(str)

print(f"Raw traffic table: {traffic_df.shape[0]:,} rows")
print(f"Unique routes: {traffic_df['route_id'].nunique()}")
print(f"Date range: {traffic_df['date'].min()} to {traffic_df['date'].max()}")

# Aggregate: daily average vehicles + max accident per route
traffic_daily = traffic_df.groupby(
    ['route_id', 'date_str']
).agg(
    avg_no_of_vehicles=('no_of_vehicles', 'mean'),
    accident=('accident', 'max')      # 1 if ANY accident occurred that day
).reset_index()

print(f"\nAggregated traffic: {traffic_daily.shape[0]:,} rows (route-day pairs)")
print(f"Columns: {traffic_daily.columns.tolist()}")
display(traffic_daily.head(3))

In [ ]:
# ---- Merge traffic onto the main dataset ----
df_merged = pd.merge(
    df_merged,
    traffic_daily,
    left_on=['route_id', 'depart_date'],
    right_on=['route_id', 'date_str'],
    how='left'
)

# Drop the redundant date_str column from traffic
if 'date_str' in df_merged.columns:
    df_merged = df_merged.drop(columns=['date_str'])

print(f"After traffic merge: {df_merged.shape}")
assert df_merged.shape[0] == schedule_df.shape[0], "Row explosion detected -- check join keys!"
print(f"  avg_no_of_vehicles nulls: {df_merged['avg_no_of_vehicles'].isnull().sum()}")
print(f"  accident nulls:           {df_merged['accident'].isnull().sum()}")

## 7. Aggregate Weather Data

Weather comes from **three perspectives** — this is the most complex part of the feature engineering:

1. **Route weather** (`routes_weather`): Weather conditions along the route itself
2. **Origin city weather** (`city_weather` filtered by `origin_id`): Weather at the departure city
3. **Destination city weather** (`city_weather` filtered by `destination_id`): Weather at the arrival city

Each source produces 6 numeric features (avg_temp, avg_wind_speed, avg_precip, avg_humidity, avg_visibility, avg_pressure) plus 1 categorical (`description` — e.g., "Clear", "Cloudy").

**Total: 6 numeric + 1 categorical per source x 3 sources = 18 numeric + 3 categorical = 21 weather features**

In [ ]:
# ---- Helper: Aggregate weather to daily level ----
def aggregate_weather_daily(weather_df, group_cols, prefix):
    """Aggregate hourly weather data to daily averages.

    Args:
        weather_df: DataFrame with hourly weather records
        group_cols: list of columns to group by (e.g., ['route_id', 'date_str'])
        prefix: str prefix for output columns (e.g., 'route', 'origin', 'dest')

    Returns:
        DataFrame with daily weather aggregates
    """
    # Custom mode function for description (most frequent weather condition)
    def custom_mode(series):
        """Return the most common value in a series."""
        mode_val = series.mode()
        return mode_val.iloc[0] if len(mode_val) > 0 else 'Unknown'

    agg_dict = {
        'temp': 'mean',
        'wind_speed': 'mean',
        'precip': 'mean',
        'humidity': 'mean',
        'visibility': 'mean',
        'pressure': 'mean',
        'description': custom_mode
    }

    daily = weather_df.groupby(group_cols).agg(agg_dict).reset_index()

    # Rename columns with prefix
    rename_map = {
        'temp': f'{prefix}_avg_temp',
        'wind_speed': f'{prefix}_avg_wind_speed',
        'precip': f'{prefix}_avg_precip',
        'humidity': f'{prefix}_avg_humidity',
        'visibility': f'{prefix}_avg_visibility',
        'pressure': f'{prefix}_avg_pressure',
        'description': f'{prefix}_description'
    }
    daily = daily.rename(columns=rename_map)

    print(f"  {prefix} weather: {daily.shape[0]:,} daily records, {daily.shape[1]} columns")
    return daily

print("Aggregating weather data to daily level...")

In [ ]:
# ---- 7a. Route Weather (routes_weather table) ----
routes_weather_df['date'] = pd.to_datetime(routes_weather_df['date'])
routes_weather_df['date_str'] = routes_weather_df['date'].dt.date.astype(str)

route_weather_daily = aggregate_weather_daily(
    routes_weather_df,
    group_cols=['route_id', 'date_str'],
    prefix='route'
)
display(route_weather_daily.head(3))

In [ ]:
# ---- 7b. Origin and Destination City Weather ----
city_weather_df['date'] = pd.to_datetime(city_weather_df['date'])
city_weather_df['date_str'] = city_weather_df['date'].dt.date.astype(str)

# Origin weather: aggregate by city_id + date
origin_weather_daily = aggregate_weather_daily(
    city_weather_df,
    group_cols=['city_id', 'date_str'],
    prefix='origin'
)

# Destination weather: same aggregation, different prefix
dest_weather_daily = aggregate_weather_daily(
    city_weather_df,
    group_cols=['city_id', 'date_str'],
    prefix='dest'
)

display(origin_weather_daily.head(3))

In [ ]:
# ---- 7c. Merge all weather features onto the main dataset ----

# Route weather: join on route_id + departure date
df_merged = pd.merge(
    df_merged,
    route_weather_daily,
    left_on=['route_id', 'depart_date'],
    right_on=['route_id', 'date_str'],
    how='left'
)
if 'date_str' in df_merged.columns:
    df_merged = df_merged.drop(columns=['date_str'])
print(f"After route weather merge:       {df_merged.shape}")

# Origin city weather: join on origin_id + departure date
df_merged = pd.merge(
    df_merged,
    origin_weather_daily,
    left_on=['origin_id', 'depart_date'],
    right_on=['city_id', 'date_str'],
    how='left'
)
for col in ['city_id', 'date_str']:
    if col in df_merged.columns:
        df_merged = df_merged.drop(columns=[col])
print(f"After origin weather merge:      {df_merged.shape}")

# Destination city weather: join on destination_id + departure date
df_merged = pd.merge(
    df_merged,
    dest_weather_daily,
    left_on=['destination_id', 'depart_date'],
    right_on=['city_id', 'date_str'],
    how='left'
)
for col in ['city_id', 'date_str']:
    if col in df_merged.columns:
        df_merged = df_merged.drop(columns=[col])
print(f"After destination weather merge: {df_merged.shape}")

# Verify no row explosion
assert df_merged.shape[0] == schedule_df.shape[0], "Row count changed after weather merges!"
print(f"\nAll weather merges complete. Row count preserved: {df_merged.shape[0]:,}")

# Check weather null counts
weather_cols = [c for c in df_merged.columns if any(
    p in c for p in ['route_avg', 'origin_avg', 'dest_avg', '_description']
)]
print(f"\nWeather feature null counts:")
for col in weather_cols:
    nulls = df_merged[col].isnull().sum()
    if nulls > 0:
        print(f"  {col:35s} {nulls:,} nulls ({nulls/len(df_merged)*100:.1f}%)")

## 8. Temporal Feature Engineering

Departure time strongly influences delay probability:
- **Rush hour departures** (7-9 AM, 5-8 PM) face heavier traffic
- **Midnight shipments** (11 PM - 5 AM) face fatigue-related risks
- **Weekend vs. weekday** patterns differ for urban vs. highway routes

We extract three temporal features:
1. `hour_of_day` — departure hour (0-23)
2. `day_of_week` — day name (Monday-Sunday)
3. `is_midnight` — binary flag: 1 if the shipment window overlaps 11 PM to 5 AM

In [ ]:
# ---- Extract temporal features ----
df_merged['hour_of_day'] = df_merged['departure_date'].dt.hour
df_merged['day_of_week'] = df_merged['departure_date'].dt.day_name()

def has_midnight_overlap(departure, arrival):
    """Check if a shipment window overlaps the midnight period (11 PM - 5 AM).

    A shipment is flagged as midnight if either:
    - Departure hour is between 23 and 5 (inclusive)
    - Arrival hour is between 23 and 5 (inclusive)
    - The shipment spans across midnight

    Args:
        departure: pd.Timestamp
        arrival: pd.Timestamp

    Returns:
        int: 1 if midnight overlap, 0 otherwise
    """
    dep_hour = departure.hour
    arr_hour = arrival.hour

    # Direct midnight hours
    if dep_hour >= 23 or dep_hour <= 5:
        return 1
    if arr_hour >= 23 or arr_hour <= 5:
        return 1
    # Spans across midnight (departs before midnight, arrives after)
    if departure.date() != arrival.date():
        return 1
    return 0

df_merged['is_midnight'] = df_merged.apply(
    lambda row: has_midnight_overlap(row['departure_date'], row['estimated_arrival']),
    axis=1
)

print("Temporal features created:")
print(f"  hour_of_day range:  {df_merged['hour_of_day'].min()} - {df_merged['hour_of_day'].max()}")
print(f"  day_of_week values: {df_merged['day_of_week'].unique().tolist()}")
print(f"  is_midnight:        {df_merged['is_midnight'].value_counts().to_dict()}")

In [ ]:
# ---- Delay rate by hour of day and day of week ----
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Delay rate by hour
hourly_delay = df_merged.groupby('hour_of_day')['delay'].mean() * 100
axes[0].bar(hourly_delay.index, hourly_delay.values, color='#2563eb', edgecolor='white')
axes[0].axhline(y=df_merged['delay'].mean()*100, color='#f43f5e', ls='--', lw=2,
                label=f"Overall: {df_merged['delay'].mean()*100:.1f}%")
axes[0].set_title('Delay Rate by Hour of Day', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Hour (0-23)')
axes[0].set_ylabel('Delay Rate (%)')
axes[0].legend()

# Delay rate by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_delay = df_merged.groupby('day_of_week')['delay'].mean().reindex(day_order) * 100
axes[1].bar(range(7), daily_delay.values, color='#10b981', edgecolor='white')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1].axhline(y=df_merged['delay'].mean()*100, color='#f43f5e', ls='--', lw=2,
                label=f"Overall: {df_merged['delay'].mean()*100:.1f}%")
axes[1].set_title('Delay Rate by Day of Week', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Delay Rate (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Midnight vs non-midnight
midnight_delay = df_merged.groupby('is_midnight')['delay'].mean() * 100
print(f"\nDelay rate comparison:")
print(f"  Non-midnight shipments: {midnight_delay.get(0, 0):.1f}%")
print(f"  Midnight shipments:     {midnight_delay.get(1, 0):.1f}%")

## 9. Exploratory Data Analysis

Now that we have all features merged, let's explore the relationships between features and the delay target.

In [ ]:
# ---- 9a. Correlation Heatmap (numeric features) ----
numeric_cols = [
    'route_avg_temp', 'route_avg_wind_speed', 'route_avg_precip',
    'route_avg_humidity', 'route_avg_visibility', 'route_avg_pressure',
    'origin_avg_temp', 'origin_avg_wind_speed', 'origin_avg_precip',
    'origin_avg_humidity', 'origin_avg_visibility', 'origin_avg_pressure',
    'dest_avg_temp', 'dest_avg_wind_speed', 'dest_avg_precip',
    'dest_avg_humidity', 'dest_avg_visibility', 'dest_avg_pressure',
    'truck_age', 'load_capacity_pounds', 'mileage_mpg',
    'age', 'experience', 'average_speed_mph',
    'avg_no_of_vehicles', 'distance', 'average_hours',
    'delay'
]

# Filter to columns that exist in the merged dataset
available_numeric = [c for c in numeric_cols if c in df_merged.columns]
corr_matrix = df_merged[available_numeric].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=False, fmt='.2f',
    cmap='RdYlBu_r', center=0, square=True,
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# Top correlations with delay
if 'delay' in corr_matrix.columns:
    delay_corr = corr_matrix['delay'].drop('delay').abs().sort_values(ascending=False)
    print("Top 10 features correlated with delay:")
    for feat, val in delay_corr.head(10).items():
        direction = "+" if corr_matrix.loc[feat, 'delay'] > 0 else "-"
        print(f"  {feat:35s} {direction}{val:.4f}")

In [ ]:
# ---- 9b. Delay Rate by Categorical Features ----
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Fuel type
if 'fuel_type' in df_merged.columns:
    fuel_delay = df_merged.groupby('fuel_type')['delay'].agg(['mean', 'count'])
    fuel_delay['mean'] = fuel_delay['mean'] * 100
    bars = axes[0, 0].bar(fuel_delay.index, fuel_delay['mean'],
                          color=['#2563eb', '#f59e0b', '#10b981', '#f43f5e'][:len(fuel_delay)],
                          edgecolor='white')
    axes[0, 0].set_title('Delay Rate by Fuel Type', fontweight='bold')
    axes[0, 0].set_ylabel('Delay Rate (%)')
    for bar, (_, row) in zip(bars, fuel_delay.iterrows()):
        axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        f'n={int(row["count"]):,}', ha='center', fontsize=9)

# Driving style
if 'driving_style' in df_merged.columns:
    style_delay = df_merged.groupby('driving_style')['delay'].agg(['mean', 'count'])
    style_delay['mean'] = style_delay['mean'] * 100
    bars = axes[0, 1].bar(style_delay.index, style_delay['mean'],
                          color=['#2563eb', '#f59e0b', '#10b981'][:len(style_delay)],
                          edgecolor='white')
    axes[0, 1].set_title('Delay Rate by Driving Style', fontweight='bold')
    axes[0, 1].set_ylabel('Delay Rate (%)')
    for bar, (_, row) in zip(bars, style_delay.iterrows()):
        axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        f'n={int(row["count"]):,}', ha='center', fontsize=9)

# Ratings
if 'ratings' in df_merged.columns:
    rating_delay = df_merged.groupby('ratings')['delay'].agg(['mean', 'count'])
    rating_delay['mean'] = rating_delay['mean'] * 100
    bars = axes[1, 0].bar(rating_delay.index.astype(str), rating_delay['mean'],
                          color='#8b5cf6', edgecolor='white')
    axes[1, 0].set_title('Delay Rate by Driver Rating', fontweight='bold')
    axes[1, 0].set_ylabel('Delay Rate (%)')
    axes[1, 0].set_xlabel('Rating')

# Gender
if 'gender' in df_merged.columns:
    gender_delay = df_merged.groupby('gender')['delay'].agg(['mean', 'count'])
    gender_delay['mean'] = gender_delay['mean'] * 100
    bars = axes[1, 1].bar(gender_delay.index, gender_delay['mean'],
                          color=['#2563eb', '#f43f5e'][:len(gender_delay)],
                          edgecolor='white')
    axes[1, 1].set_title('Delay Rate by Driver Gender', fontweight='bold')
    axes[1, 1].set_ylabel('Delay Rate (%)')
    for bar, (_, row) in zip(bars, gender_delay.iterrows()):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        f'n={int(row["count"]):,}', ha='center', fontsize=9)

plt.suptitle('Delay Rate by Categorical Features', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---- 9c. Distributions of Key Continuous Features ----
continuous_features = {
    'truck_age': 'Truck Age (years)',
    'distance': 'Route Distance (km)',
    'average_hours': 'Average Route Hours',
    'avg_no_of_vehicles': 'Daily Avg Vehicles on Route',
    'experience': 'Driver Experience (years)',
    'average_speed_mph': 'Driver Avg Speed (mph)'
}

# Filter to available columns
available = {k: v for k, v in continuous_features.items() if k in df_merged.columns}
n_plots = len(available)
n_cols = 3
n_rows = (n_plots + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, (col, label) in enumerate(available.items()):
    # Split by delay status
    on_time = df_merged[df_merged['delay'] == 0][col].dropna()
    delayed = df_merged[df_merged['delay'] == 1][col].dropna()

    axes[i].hist(on_time, bins=30, alpha=0.6, color='#2563eb', label='On Time', edgecolor='white')
    axes[i].hist(delayed, bins=30, alpha=0.6, color='#f43f5e', label='Delayed', edgecolor='white')
    axes[i].set_title(label, fontweight='bold', fontsize=11)
    axes[i].legend(fontsize=8)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Feature Distributions: On Time vs Delayed', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Assemble Final Feature Matrix

Now we select the **36 features** (27 continuous + 9 categorical) plus the target. This is the analysis-ready dataset that Lab D will consume for model training.

**27 Continuous Features:**
- Route weather (6): route_avg_temp, route_avg_wind_speed, route_avg_precip, route_avg_humidity, route_avg_visibility, route_avg_pressure
- Origin weather (6): origin_avg_temp, origin_avg_wind_speed, origin_avg_precip, origin_avg_humidity, origin_avg_visibility, origin_avg_pressure
- Destination weather (6): dest_avg_temp, dest_avg_wind_speed, dest_avg_precip, dest_avg_humidity, dest_avg_visibility, dest_avg_pressure
- Vehicle metrics (3): truck_age, load_capacity_pounds, mileage_mpg
- Driver metrics (3): age, experience, average_speed_mph
- Traffic (1): avg_no_of_vehicles
- Route (2): distance, average_hours

**9 Categorical Features:**
- route_description, origin_description, dest_description
- accident, fuel_type, gender, driving_style, ratings, is_midnight

**Target:** delay (0/1)

In [ ]:
# ---- Define the 36 feature columns + target ----
CONTINUOUS_FEATURES = [
    # Route weather (6)
    'route_avg_temp', 'route_avg_wind_speed', 'route_avg_precip',
    'route_avg_humidity', 'route_avg_visibility', 'route_avg_pressure',
    # Origin weather (6)
    'origin_avg_temp', 'origin_avg_wind_speed', 'origin_avg_precip',
    'origin_avg_humidity', 'origin_avg_visibility', 'origin_avg_pressure',
    # Destination weather (6)
    'dest_avg_temp', 'dest_avg_wind_speed', 'dest_avg_precip',
    'dest_avg_humidity', 'dest_avg_visibility', 'dest_avg_pressure',
    # Vehicle metrics (3)
    'truck_age', 'load_capacity_pounds', 'mileage_mpg',
    # Driver metrics (3)
    'age', 'experience', 'average_speed_mph',
    # Traffic (1)
    'avg_no_of_vehicles',
    # Route (2)
    'distance', 'average_hours'
]

CATEGORICAL_FEATURES = [
    'route_description', 'origin_description', 'dest_description',
    'accident', 'fuel_type', 'gender', 'driving_style', 'ratings',
    'is_midnight'
]

TARGET = 'delay'

ALL_FEATURES = CONTINUOUS_FEATURES + CATEGORICAL_FEATURES
FINAL_COLUMNS = ALL_FEATURES + [TARGET]

print(f"Feature matrix specification:")
print(f"  Continuous features: {len(CONTINUOUS_FEATURES)}")
print(f"  Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"  Target: {TARGET}")
print(f"  Total columns: {len(FINAL_COLUMNS)} (36 features + 1 target)")

In [ ]:
# ---- Select and validate the feature matrix ----
# Check which columns are available
missing_cols = [c for c in FINAL_COLUMNS if c not in df_merged.columns]
if missing_cols:
    print(f"WARNING: Missing columns: {missing_cols}")
    print("These will be handled below.")

# Select available columns
available_cols = [c for c in FINAL_COLUMNS if c in df_merged.columns]
df_features = df_merged[available_cols].copy()

print(f"Feature matrix shape: {df_features.shape}")
print(f"Expected: ~12,308 rows x {len(FINAL_COLUMNS)} columns")

# ---- Handle missing values ----
print(f"\nNull counts before handling:")
null_counts = df_features.isnull().sum()
null_cols = null_counts[null_counts > 0]
if len(null_cols) > 0:
    for col, count in null_cols.items():
        print(f"  {col:35s} {count:,} nulls ({count/len(df_features)*100:.1f}%)")

    # Fill strategy: numeric with median, categorical with mode/'Unknown'
    for col in CONTINUOUS_FEATURES:
        if col in df_features.columns and df_features[col].isnull().any():
            median_val = df_features[col].median()
            df_features[col] = df_features[col].fillna(median_val)
            print(f"  Filled {col} with median: {median_val:.2f}")

    for col in CATEGORICAL_FEATURES:
        if col in df_features.columns and df_features[col].isnull().any():
            df_features[col] = df_features[col].fillna('Unknown')
            print(f"  Filled {col} with 'Unknown'")
else:
    print("  No nulls found -- clean dataset!")

# Final null check
remaining_nulls = df_features.isnull().sum().sum()
print(f"\nRemaining nulls after handling: {remaining_nulls}")
assert remaining_nulls == 0, "Still have nulls -- investigate!"

print(f"\nFinal feature matrix: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns")

In [ ]:
# ---- Summary statistics of the final feature matrix ----
print("=== Final Feature Matrix Summary ===\n")

print("Continuous features:")
display(df_features[CONTINUOUS_FEATURES].describe().round(2))

print("\nCategorical features:")
for col in CATEGORICAL_FEATURES:
    if col in df_features.columns:
        unique = df_features[col].nunique()
        top = df_features[col].value_counts().index[0]
        print(f"  {col:25s}  unique: {unique:5d}  most common: {top}")

print(f"\nTarget distribution:")
print(f"  {df_features[TARGET].value_counts().to_dict()}")

## 11. Save Artifacts

We save two files:
1. **`final_features.csv`** — the complete feature matrix (consumed by Lab D for model training)
2. **`feature_metadata.json`** — column names, types, and counts (useful for pipeline automation)

Both are saved locally and uploaded to S3 under `data/processed/`.

In [ ]:
# ---- Save final_features.csv ----
csv_path = os.path.join(OUTPUT_DIR, 'final_features.csv')
df_features.to_csv(csv_path, index=False)
file_size_mb = os.path.getsize(csv_path) / (1024 * 1024)
print(f"Saved: {csv_path}")
print(f"  Shape: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns")
print(f"  Size:  {file_size_mb:.1f} MB")

In [ ]:
# ---- Save feature_metadata.json ----
metadata = {
    'project': 'Truck Delay Classification',
    'module': 'M3',
    'lab': 'Lab C -- EDA & Feature Engineering',
    'created_by': 'FreshBasket ML Team (Priya)',
    'num_rows': int(df_features.shape[0]),
    'num_features': int(df_features.shape[1] - 1),  # exclude target
    'target': TARGET,
    'continuous_features': CONTINUOUS_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'feature_count': {
        'continuous': len(CONTINUOUS_FEATURES),
        'categorical': len(CATEGORICAL_FEATURES),
        'total': len(ALL_FEATURES)
    }
}

json_path = os.path.join(OUTPUT_DIR, 'feature_metadata.json')
with open(json_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {json_path}")
print(f"  Features: {metadata['feature_count']['total']} ({metadata['feature_count']['continuous']} continuous + {metadata['feature_count']['categorical']} categorical)")
print(f"  Target: {metadata['target']}")

In [ ]:
# ---- Upload to S3 ----
# Uncomment the lines below to upload to your S3 bucket
# import boto3
# s3_client = boto3.client('s3')
#
# s3_client.upload_file(csv_path, S3_BUCKET, 'data/processed/final_features.csv')
# print(f"Uploaded to s3://{S3_BUCKET}/data/processed/final_features.csv")
#
# s3_client.upload_file(json_path, S3_BUCKET, 'data/processed/feature_metadata.json')
# print(f"Uploaded to s3://{S3_BUCKET}/data/processed/feature_metadata.json")

print("To upload to S3, uncomment the boto3 code above and run this cell.")
print(f"Target location: s3://{S3_BUCKET}/data/processed/")
print()
print("=== Artifact Summary ===")
print(f"  Local:  {csv_path}  ({file_size_mb:.1f} MB)")
print(f"  Local:  {json_path}")
print(f"  S3:     s3://{S3_BUCKET}/data/processed/final_features.csv")
print(f"  S3:     s3://{S3_BUCKET}/data/processed/feature_metadata.json")

---

## 12. Conclusion & Next Steps

### What We Covered

| Task | Key Takeaway |
|------|-------------|
| **RDS connection** | SQLAlchemy engine connects notebooks to PostgreSQL; keep credentials out of code in production |
| **Table profiling** | Always check shape, dtypes, and nulls before merging — prevents silent data loss |
| **Core merges** | Left joins from the schedule backbone preserve the 12,308-row structure |
| **Traffic aggregation** | 2.6M hourly rows aggregated to daily route averages; verified no row explosion |
| **Weather aggregation** | Three weather perspectives (route, origin city, destination city) produce 21 features |
| **Temporal features** | Hour-of-day and midnight flag capture time-dependent delay patterns |
| **EDA insights** | Correlation heatmap and categorical breakdowns reveal which features drive delay |
| **Feature matrix** | 36 features + 1 target, zero nulls, saved as CSV + JSON metadata |

### Try on Your Own

1. Add `chanceofrain`, `chanceoffog`, `chanceofsnow`, `chanceofthunder` as additional weather features — do they improve signal?
2. Create interaction features: `distance * avg_no_of_vehicles` (congestion-adjusted distance)
3. Try `departure_month` as a seasonal feature — does delay rate vary by season in India?
4. Compute `speed_ratio = average_speed_mph / (distance / average_hours)` — how well does the driver's usual speed match the route's expected pace?
5. Explore multicollinearity: origin and destination weather features may be highly correlated — would PCA help?

### Artifact Chain

```
Lab C output -->  data/processed/final_features.csv        (12,308 x 37)
                  data/processed/feature_metadata.json
                        |
                        v
Lab D input  -->  Load final_features.csv
                  Train/test split, model training, MLflow logging
```

---

**Next Lab:** Module 3, Lab D — Model Training & MLflow Experiment Tracking

In [ ]:
# ---- Final Dataset Statistics ----
print("=" * 60)
print("  FINAL DATASET STATISTICS")
print("=" * 60)
print(f"  Rows:                {df_features.shape[0]:,}")
print(f"  Features:            {df_features.shape[1] - 1}")
print(f"  Target:              delay (binary 0/1)")
print(f"  Continuous features: {len(CONTINUOUS_FEATURES)}")
print(f"  Categorical features:{len(CATEGORICAL_FEATURES)}")
print(f"  Null values:         {df_features.isnull().sum().sum()}")
print(f"  Delay rate:          {df_features['delay'].mean()*100:.1f}%")
print(f"  Saved to:            {csv_path}")
print("=" * 60)
print("\nLab C complete. Proceed to Lab D for model training.")